In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1. Воспроизводимость
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

set_seed(42)

# 2. Настройки
SUBMISSIONS_DIR = "submissions"
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
TIME_LIMIT = 3600 # 1 час - минимум для best_quality
GLOBAL_CEILING = 306210000.0 

# 3. Загрузка данных
print("Loading data...")
train = pd.read_parquet('data/train_solo_track.parquet')
test = pd.read_parquet('data/test_solo_track.parquet')

# Для best_quality берем последние 14 дней (чтобы влезть в RAM/VRAM при стэкинге)
cutoff_date = train['timestamp'].max() - pd.Timedelta(days=14)
train = train[train['timestamp'] > cutoff_date].copy()

# 4. FEATURE ENGINEERING (Наши лучшие находки)
def add_features(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    # Календарь
    df['is_working_day'] = df['timestamp'].dt.dayofweek.map({0:1, 1:1, 2:1, 3:1, 4:1, 5:0, 6:0}).astype(float)
    df.loc[df['timestamp'].dt.date == pd.to_datetime('2025-11-01').date(), 'is_working_day'] = 1.0
    # Физика
    if 'status_1' in df.columns:
        df['total_load'] = df['status_1'] + df['status_2'] + df['status_3']
    return df

train = add_features(train)

# Утренние якоря (10:30)
anchors = train[
    (train['timestamp'].dt.hour == 10) & (train['timestamp'].dt.minute == 30)
][['route_id', 'date', 'target_1h', 'total_load']].rename(
    columns={'target_1h': 'anchor_target', 'total_load': 'anchor_load'}
)
train = train.merge(anchors, on=['route_id', 'date'], how='left')
train['anchor_target'] = train.groupby('route_id')['anchor_target'].ffill().fillna(0)
train['anchor_load'] = train.groupby('route_id')['anchor_load'].ffill().fillna(0)

# Тест
test = add_features(test)
last_morning = train[train['timestamp'] == train['timestamp'].max()][['route_id', 'anchor_target', 'anchor_load']]
test = test.merge(last_morning, on='route_id', how='left')

# 5. ПОДГОТОВКА ДАННЫХ
train_ag = train.rename(columns={'route_id': 'item_id', 'target_1h': 'target'}).drop(columns=['date'])
test_ag = test.rename(columns={'route_id': 'item_id'}).drop(columns=['date'])

train_data = TimeSeriesDataFrame.from_data_frame(
    train_ag, id_column="item_id", timestamp_column="timestamp"
).sort_index()

known_covariates_list = ['is_working_day', 'anchor_target', 'anchor_load']

# 6. СПЕЦИАЛИЗИРОВАННАЯ ВАЛИДАЦИЯ (Peak Hours)
val_days = 3 # Для best_quality хватит 3 дней валидации
val_cutoff = train['timestamp'].max() - pd.Timedelta(days=val_days)
tuning_data_full = train_data.slice_by_time(val_cutoff, train['timestamp'].max())
tuning_times = tuning_data_full.index.get_level_values('timestamp')
tuning_data = tuning_data_full[(tuning_times.hour >= 11) & (tuning_times.hour <= 14)]

# 7. ОБУЧЕНИЕ (BEST QUALITY)
predictor = TimeSeriesPredictor(
    prediction_length=8,
    target="target",
    eval_metric="WAPE",
    freq="30min",
    known_covariates_names=known_covariates_list
)

print(f"\n🚀 Starting AutoGluon BEST_QUALITY Fit (Time limit: {TIME_LIMIT}s)...")
predictor.fit(
    train_data,
    tuning_data=tuning_data,
    time_limit=TIME_LIMIT,
    presets="best_quality", # МАКСИМАЛЬНОЕ КАЧЕСТВО
    random_seed=42
)

# 8. ИНФЕРЕНС
print("\nStarting Inference...")
future_covariates = TimeSeriesDataFrame.from_data_frame(
    test_ag[['item_id', 'timestamp'] + known_covariates_list],
    id_column="item_id", timestamp_column="timestamp"
).sort_index()

predictions = predictor.predict(train_data, known_covariates=future_covariates)

# 9. ПОСТ-ПРОЦЕССИНГ (Клиппинг)
predictions = predictions.reset_index()
predictions['y_pred_raw'] = predictions['mean'].clip(0)

# Глобальный клиппинг по потолку склада
time_sums = predictions.groupby('timestamp')['y_pred_raw'].transform('sum')
predictions['y_pred'] = np.where(
    time_sums > GLOBAL_CEILING,
    predictions['y_pred_raw'] * (GLOBAL_CEILING / time_sums),
    predictions['y_pred_raw']
)

# 10. САБМИТ
test_orig = pd.read_parquet('data/test_solo_track.parquet')
test_orig['timestamp'] = pd.to_datetime(test_orig['timestamp'])
predictions = predictions.rename(columns={'item_id': 'route_id'})
submission = test_orig.merge(predictions[['route_id', 'timestamp', 'y_pred']], on=['route_id', 'timestamp'], how='left')

sub_path = f"{SUBMISSIONS_DIR}/autogluon_best_quality_sub.csv"
submission[['id', 'y_pred']].to_csv(sub_path, index=False)

print(f"\n🏆 Done! Best Quality submission saved to {sub_path}")
print(f"Best model found: {predictor.model_best}")
leaderboard = predictor.leaderboard(train_data, silent=True)
display(leaderboard.head(10))